# Module 7: Credit Derivatives Verification
This notebook verifies the ISDA Standard CDS model and Merton Jump-Diffusion model implementation.

In [1]:
import sys
import os
import datetime
sys.path.insert(0, os.path.abspath(".."))

from src.credit.cds import isda_upfront_charge, isda_par_spread
from src.credit.structural import MertonJumpDiffusionModel
from src.utils.plotting import set_theme

set_theme()


## ISDA Standard CDS Model
Verify exact IMM dates upfront/spread conversion.

In [2]:
valuation_date = datetime.date(2023, 9, 15)
effective_date = datetime.date(2023, 9, 16)
maturity_date = datetime.date(2028, 12, 20) # 5-year CDS

def curve(t): return 0.03 # flat 3% curve
recovery_rate = 0.40
notional = 10_000_000
standard_coupon = 0.01
quoted_par_spread = 0.0120

upfront = isda_upfront_charge(
    valuation_date, effective_date, maturity_date,
    quoted_par_spread, standard_coupon, curve, recovery_rate, notional
)
print(f"Quoted Par Spread: {quoted_par_spread*10000:.0f} bps")
print(f"Standard Coupon: {standard_coupon*10000:.0f} bps")
print(f"Upfront Charge (USD): ${upfront:,.2f}")

implied_par = isda_par_spread(
    valuation_date, effective_date, maturity_date,
    upfront, standard_coupon, curve, recovery_rate, notional
)
print(f"Implied Par Spread: {implied_par*10000:.2f} bps")


Quoted Par Spread: 120 bps
Standard Coupon: 100 bps
Upfront Charge (USD): $93,140.45
Implied Par Spread: 120.00 bps


## Merton Jump-Diffusion Model
Calculate equity and debt value under the jump-diffusion model.

In [3]:
mjd = MertonJumpDiffusionModel(
    V=100_000_000.0,
    D=80_000_000.0,
    T=1.0,
    r=0.03,
    sigma_V=0.20,
    lambda_j=0.5,
    mu_j=-0.1,
    sigma_j=0.3
)
equity = mjd.equity_value()
debt = mjd.debt_value()

print(f"Asset Value: ${mjd.V:,.2f}")
print(f"Face Value of Debt: ${mjd.D:,.2f}")
print(f"MJD Equity Value: ${equity:,.2f}")
print(f"MJD Debt Value: ${debt:,.2f}")


Asset Value: $100,000,000.00
Face Value of Debt: $80,000,000.00
MJD Equity Value: $25,216,061.71
MJD Debt Value: $74,783,938.29
